<a href="https://colab.research.google.com/github/siddharthsinh-dev/iu-bsc-thesis-llm-comparison/blob/main/notebooks/exp1_a_finbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Thesis**: A Comparative Study of Large Language Models for Financial Sentiment Analysis and Their Predictive Potential for Short-Term Stock Price Movement

**Component: Experiment 1** - Model A : FinBERT (ProsusAI/finbert)

**Description:** This notebook evaluates FinBERT on the Financial PhraseBank dataset under zero-shot conditions. It generates sentiment predictions and computes accuracy, precision, recall, F1-score (macro-averaged), and confusion matrix.

Select T4 GPU as runtime.

In [ ]:
# Install required libraries — Experiment 1a: FinBERT

!pip install -q transformers
!pip install -q pandas scikit-learn matplotlib seaborn

print("Libraries installed successfully.")

In [ ]:
# Import required libraries — Experiment 1a: FinBERT

import pandas as pd
import numpy as np
import torch
import warnings

from transformers import pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU detected    : {torch.cuda.get_device_name(0)}")

Add the API Key from Hugging Face using "Add New Secret" in Google Colab

In [ ]:
# Connect to Hugging Face
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

print("Hugging Face login successful.")

# Load Financial PhraseBank — full dataset
file_path = hf_hub_download(
    repo_id="takala/financial_phrasebank",
    filename="sentences_50agree/train-00000-of-00001.parquet",
    repo_type="dataset",
    revision="0dd3028d70cbd18ded8887e65e83343b03a50482",
    token=hf_token
)

df_fpb = pd.read_parquet(file_path)

label_map = {0: "negative", 1: "neutral", 2: "positive"}
df_fpb["sentiment"] = df_fpb["label"].map(label_map)

true_labels = df_fpb["sentiment"].tolist()
texts = df_fpb["sentence"].tolist()

print(f"\nFinancial PhraseBank loaded.")
print(f"Total sentences : {len(df_fpb)}")
print(f"Label distribution:")
print(df_fpb["sentiment"].value_counts())

In [ ]:
# Load FinBERT
# Model: ProsusAI/finbert
# Domain-specific financial sentiment model

print("Loading FinBERT...")

finbert = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1
)

print("FinBERT loaded successfully.")

In [ ]:
# Run FinBERT on full Financial PhraseBank (4,846 sentences)

print("Running FinBERT on 4,846 sentences...")

finbert_results = finbert(texts, batch_size=32)
finbert_preds = [r["label"].lower() for r in finbert_results]

print(f"Done. Total predictions: {len(finbert_preds)}")
print(f"\nPrediction distribution:")
print(pd.Series(finbert_preds).value_counts())

In [ ]:
# Evaluate FinBERT performance

labels_order = ["positive", "negative", "neutral"]

acc  = accuracy_score(true_labels, finbert_preds)
prec = precision_score(true_labels, finbert_preds, average="macro", labels=labels_order)
rec  = recall_score(true_labels, finbert_preds, average="macro", labels=labels_order)
f1   = f1_score(true_labels, finbert_preds, average="macro", labels=labels_order)

print("=" * 45)
print("FinBERT - Experiment 1 Results")
print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)

print("\nDetailed Classification Report:")
print(classification_report(true_labels, finbert_preds, labels=labels_order))

In [ ]:
# Save FinBERT results and predictions

all_results = []

finbert_scores = {
    "model": "FinBERT",
    "accuracy": round(acc, 4),
    "precision": round(prec, 4),
    "recall": round(rec, 4),
    "f1_score": round(f1, 4)
}

all_results.append(finbert_scores)

# Save predictions to dataframe
df_fpb["finbert_pred"] = finbert_preds

print("FinBERT results saved.")
print(pd.DataFrame(all_results))

In [ ]:
# Save FinBERT predictions to Google Drive

from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/Thesis_Data", exist_ok=True)

df_fpb.to_csv("/content/drive/MyDrive/Thesis_Data/exp1_finbert_preds.csv", index=False)

print("FinBERT predictions saved to Google Drive.")

In [ ]:
# Confusion Matrix — FinBERT Experiment 1

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

labels_order = ["positive", "negative", "neutral"]

cm = confusion_matrix(true_labels, finbert_preds, labels=labels_order)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order,
            yticklabels=labels_order)
plt.xlabel("Predicted Label", fontsize=12)
plt.ylabel("True Label", fontsize=12)
plt.title("FinBERT — Confusion Matrix (Experiment 1)", fontsize=12)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/Thesis_Data/fig_cm_finbert.png",
            dpi=300, bbox_inches="tight")
plt.show()
print("Confusion matrix saved to Google Drive.")